# Reconhecimento de Padrões — Random Forest
## Desafio: reprodução de um artigo científico
Bruno Bogaz Zarpelão

### Artigo
J. Qin, L. Chen, Y. Liu, C. Liu, C. Feng, B. Chen, "A Machine Learning Methodology for Diagnosing Chronic Kidney Disease", *IEEE Access*, vol. 8, pp. 20991–21002, 2020. DOI: 10.1109/ACCESS.2019.2963053 (acesso aberto).

Os autores propõem um método para diagnosticar doença renal crônica (DRC) a partir de exames clínicos de 400 pacientes. O conjunto de dados tem muitos valores faltantes, que os autores preenchem com uma imputação baseada em vizinhos mais próximos (KNN) antes de treinar os classificadores. Entre os modelos testados, a random forest obtém a melhor acurácia individual: 99,75%.

### Objetivo do desafio
Reproduzir o pipeline que leva à random forest (leitura e limpeza dos dados, codificação, imputação e avaliação) e analisar criticamente as escolhas metodológicas dos autores.

**Escopo:** seções II e III-A do artigo, com foco na random forest. Da seção II-C, entra apenas a importância das variáveis calculada pela random forest (Figura 2). Ficam de fora a regressão por subconjuntos ótimos (*optimal subset regression*), os outros classificadores e o modelo integrado (seções III-B, III-C e IV).

### Como trabalhar
O exercício é individual, mas conversar com colegas é bem-vindo. Vocês podem usar IA generativa para gerar código, mas quem responde pela análise são vocês.

    ### Termos e ferramentas
* **Imputação:** preencher valores faltantes com valores estimados, para que os modelos possam ser treinados com todas as amostras.
* **Imputação por KNN:** para uma amostra com valores faltantes, procuram-se as amostras mais parecidas com ela (as vizinhas mais próximas) e usam-se os valores dessas vizinhas para preencher os faltantes. A seção II-B descreve a versão usada no artigo.
* **Validação cruzada estratificada:** a validação cruzada em que cada dobra mantém a proporção das classes do conjunto original. No scikit-learn, `StratifiedKFold`.
* **`Pipeline`:** encadeia etapas, por exemplo imputação e classificador, em um único objeto do scikit-learn. Quando um `Pipeline` é avaliado com validação cruzada (`cross_val_score`, `cross_val_predict`), todas as etapas são ajustadas apenas com os dados de treino de cada dobra e depois aplicadas aos dados de teste daquela dobra.
* **Formato ARFF:** formato de arquivo de texto usado pela ferramenta Weka. Tem um cabeçalho que descreve as variáveis e, depois da linha `@data`, uma amostra por linha, com os valores separados por vírgula. Valores faltantes aparecem como `?`.

### Dados
Use o arquivo `chronic_kidney_disease_full.arff`, do conjunto *Chronic Kidney Disease* do UCI Machine Learning Repository, citado pelos autores (seção II-A). A Tabela 1 do artigo descreve as 24 variáveis e a classe (`ckd`: com DRC; `notckd`: sem DRC).

### Etapas
1. **Leitura do artigo.** Descreva, com suas palavras, as etapas do pipeline que levam à random forest (seções II-A, II-B e III-A): dados, codificação, imputação, modelo e forma de avaliação. Anote os resultados da random forest que você vai tentar reproduzir (Tabelas 5 e 10).
2. **Leitura e limpeza dos dados.** Leia o arquivo e gere uma tabela com a porcentagem de valores faltantes de cada variável. Os valores batem com a Tabela 1 do artigo?
3. **Codificação.** Codifique as variáveis como descrito na seção II-B.
4. **Imputação por KNN.** Implemente a imputação descrita na seção II-B, para os cinco valores de K usados pelos autores. O scikit-learn tem o `KNNImputer`: leia a documentação e compare com o procedimento do artigo. Ele faz a mesma coisa? Que diferenças importam para estes dados?
5. **Random forest.** Reproduza a avaliação da random forest descrita na seção III-A e calcule acurácia, sensibilidade, especificidade e F1-score, com `ckd` como classe positiva. Compare com as Tabelas 5 e 10. Como o conjunto tem 400 pacientes, quantos pacientes classificados errado correspondem às acurácias de 99,75% e de 99,50% que aparecem nas tabelas do artigo?
6. **Importância das variáveis.** Reproduza a Figura 2 (importância das variáveis pela random forest, no conjunto imputado com K = 9). As variáveis que os autores descartam aparecem com importância zero? Remover essas variáveis muda o desempenho da random forest?
7. **Análise da metodologia.**
   1. **Momento da imputação.** Em que momento os autores fazem a imputação em relação à validação cruzada? Por que isso pode ser um problema? Refaça a avaliação com a imputação feita dentro da validação cruzada e compare.
   2. **Tamanho das diferenças.** Os autores sorteiam as 4 dobras uma única vez (semente 1234). Repita a avaliação da etapa 5 várias vezes (por exemplo, 10), mudando a cada vez a semente usada para sortear as dobras e mantendo todo o resto igual. Quanto o resultado varia? Com base nisso, analise as comparações feitas pelos autores entre valores de K, entre a imputação por KNN e a imputação por média e moda (Tabela 5; teste também essa imputação) e entre a random forest e os modelos de outros trabalhos (Tabela 11).
   3. **Distância usada na imputação.** Os autores padronizam as variáveis antes de calcular a distância usada na imputação? Observe as escalas das variáveis. Que variáveis, na prática, determinam quais amostras são as vizinhas mais próximas?
   4. **Amostras usadas como vizinhas.** A imputação do artigo busca vizinhas apenas entre as amostras completas. De que classe são essas amostras? Compare os valores imputados nos pacientes com DRC com os valores observados em cada classe.
8. **Conclusão.** O resultado da random forest se sustenta? Que mudanças você faria no pipeline dos autores?

## Respostas

A implementação e as tabelas de resultados estão no [resultado.ipynb](./resultado.ipynb). Em cada etapa, indico a seção correspondente.

### 1. Leitura do artigo

**Complemento para revisar.** O artigo usa 400 amostras, com 24 preditores e a classe `ckd` ou `notckd`. Os autores codificam as categorias binárias como 0 e 1, tratam `sg`, `al` e `su` como numéricas e imputam os valores ausentes por KNN. Para cada amostra incompleta, procuram vizinhos entre as amostras completas e usam a mediana nas variáveis numéricas e a moda nas categóricas. Depois, avaliam uma random forest de 500 árvores com quatro dobras estratificadas.

Na Tabela 5, a referência de acurácia para a random forest é de 99,75% com `K = 3, 5 e 9` e de 99,50% com `K = 7 e 11`. A Tabela 10 apresenta as médias de dez rodadas. Para `K = 9`, as referências são 99,65% de acurácia, 99,44% de sensibilidade, 100% de especificidade e 99,72% de F1.

**No resultado.ipynb.** Seções “Variáveis e codificação”, “Imputação KNN com e sem padronização” e “Replicação da Tabela 5” e “Replicação da Tabela 10”.

### 2. Leitura e limpeza dos dados

Os percentuais de valores ausentes são iguais aos da Tabela 1 do artigo nas 25 colunas, incluindo a classe.

**No resultado.ipynb.** Seção “Comparação dos valores faltantes”.

### 3. Codificação

A codificação foi feita no notebook. As categorias binárias foram mapeadas para 0 e 1, e `sg`, `al` e `su` foram convertidas para números, preservando seus valores originais. A classe foi separada dos preditores antes da imputação.

**No resultado.ipynb.** Seção “Variáveis e codificação” e células seguintes de codificação e separação da classe.

### 4. Imputação por KNN

O procedimento foi executado para `K = 3, 5, 7, 9 e 11`, com e sem padronização. O método do artigo e o `KNNImputer` do scikit-learn usam vizinhos, mas diferem em três pontos.

- **Variáveis binárias.** Para uma coluna como `rbc`, se os três vizinhos têm valores `[0, 1, 0]`, o método do artigo usa a moda e preenche o faltante com `0`. Já o `KNNImputer`, com pesos uniformes, usa a média e devolve aproximadamente `0,33`. Esse valor representa a proporção de vizinhos com código `1`, mas fica fora das categorias permitidas para `rbc`. Um ajuste possível seria atribuir `1` quando a média fosse maior que 0,5 e `0` quando fosse menor. Em caso de empate, poderia ser adotado `0`.
- **Seleção dos vizinhos.** O artigo utiliza apenas pacientes com todos os preditores preenchidos. O `KNNImputer` também pode usar pacientes incompletos que tenham valor na coluna a imputar. Por exemplo, para preencher `rbc`, um paciente com `rbc` observado pode ser vizinho mesmo que sua idade esteja faltando.
- **Cálculo da distância.** O `KNNImputer` compara os pacientes usando as colunas observadas em ambos e ajusta a distância pela quantidade dessas colunas.

Com `K = 9` e a mesma padronização, os métodos diferiram em 935 das 1.012 células imputadas. O scikit-learn produziu 180 valores fracionários nas colunas binárias.

**No resultado.ipynb.** Seções “Imputação KNN com e sem padronização” e “Comparação com o KNNImputer do scikit-learn”.

### 5. Random forest

Os resultados ficaram próximos aos do artigo. Na replicação da Tabela 5, a acurácia foi de 99,75% para os cinco valores de `K`. Isso coincide com o artigo para `K = 3, 5 e 9` e fica 0,25 ponto percentual acima para `K = 7 e 11`.

Na replicação da Tabela 10, com `K = 9` e `ckd` como classe positiva, as médias foram de 99,65% de acurácia, 99,44% de sensibilidade, 100% de especificidade e 99,72% de F1. Os quatro valores coincidiram com os do artigo a duas casas decimais. As pequenas diferenças nas outras configurações podem estar relacionadas às implementações e às divisões usadas. O artigo usou R, e a replicação foi feita em Python com scikit-learn.

Com 400 pacientes, **99,75% corresponde a um erro**, enquanto **99,50% corresponde a dois erros** em uma rodada de validação.

**No resultado.ipynb.** Seções “Replicação da Tabela 5”, “Replicação da Tabela 10” e “Matriz de confusão nas quatro dobras”.

### 6. Importância das variáveis

Na replicação da Figura 2, com `K = 9`, só `cad` apresentou importância exatamente zero. As outras duas variáveis apontadas no artigo ficaram muito próximas de zero.

| Variável | Importância Gini |
| --- | ---: |
| `pcc` | 0,000201 |
| `ba` | 0,000291 |
| `cad` | 0,000000 |

Remover apenas `cad` ou as três variáveis manteve a acurácia de 99,75% e um erro nas quatro dobras usadas. As importâncias da replicação estão na escala normalizada do scikit-learn.

**No resultado.ipynb.** Seção “Importância das variáveis na random forest” e subseção “Desempenho com e sem as variáveis descartadas”.

### 7. Análise da metodologia

#### 7.1. Momento da imputação

No artigo, a imputação é feita antes da validação cruzada. Isso pode causar vazamento porque pacientes que depois irão para o teste podem servir como vizinhos para preencher valores do treino. Na versão padronizada, os pacientes do teste também influenciam a média e o desvio-padrão usados nas distâncias.

Na comparação, fiz a imputação dentro de cada dobra, usando apenas o treino para obter os vizinhos e calcular a padronização. Com `K = 9` e as mesmas quatro divisões, as duas versões tiveram as mesmas previsões e 99,75% de acurácia, com um falso negativo. Nesse teste, corrigir o vazamento manteve o resultado.

**No resultado.ipynb.** Seção “Possível vazamento de dados na validação cruzada”.

#### 7.2. Tamanho das diferenças

**Complemento para revisar.** Em dez rodadas, variando apenas a semente da divisão, a amplitude da acurácia foi de 0,25 a 0,50 ponto percentual para o KNN. Com `K = 9`, a acurácia ficou entre 99,50% e 99,75%. Essa variação é da mesma ordem que as diferenças entre os valores de `K` na Tabela 5.

A imputação por média e moda variou entre 98,50% e 99,50%. Nas mesmas divisões, `K = 9` teve acurácia 0,65 ponto percentual maior em média, com diferenças entre 0 e 1,25 ponto percentual. Ao comparar com a Tabela 11, a média da replicação com `K = 9` foi de 99,725%, próxima, por exemplo, aos 99,70% da SVM [27] e aos 99,75% da MLP [29]. Essas diferenças devem ser interpretadas levando em conta a variação das dobras e os protocolos próprios dos outros trabalhos.

**No resultado.ipynb.** Seção “7.2 — Variação entre divisões das dobras” e subseção “Referências da Tabela 11”.

#### 7.3. Distância usada na imputação

Não ficou claro no artigo se foi feita a padronização antes da imputação KNN. Como o método usa a distância euclidiana, a **escala** das variáveis pode alterar quais pacientes são selecionados como vizinhos. Nas escalas originais, `wbcc`, `bgr` e `bu` somaram 93,29% da contribuição média à distância quadrática. A padronização distribuiu essa contribuição entre mais variáveis. Com `K = 9`, 662 valores imputados diferiram entre as duas versões.

**No resultado.ipynb.** Seções “Imputação KNN com e sem padronização” e “7.3 — Quanto cada variável contribui para a distância”.

#### 7.4. Amostras usadas como vizinhas

| Classe | Total na base | Amostras completas | Participação entre as amostras completas |
| --- | ---: | ---: | ---: |
| `ckd` | 250 | 43 | 27,22% |
| `notckd` | 150 | 115 | 72,78% |

Os pacientes com DRC representam 27,22% das amostras completas. Dentro da classe `ckd`, 43 dos 250 pacientes (17,2%) têm todos os preditores preenchidos. Os outros 207 (82,8%) tiveram ao menos um campo imputado pela mediana ou pela moda.

**Complemento para revisar.** Para `rbc`, o código 1 apareceu em 56,1% dos valores observados em `ckd`, em 100% dos observados em `notckd` e em 81,1% dos imputados em `ckd`. Para `hemo`, as medianas foram de 10,9, 15,0 e 14,2, respectivamente. Nesses exemplos, o valor imputado ficou mais próximo do grupo `notckd`. A composição dos candidatos a vizinhos é uma possível influência sobre o preenchimento.

**No resultado.ipynb.** Seção “7.4 — Classes dos candidatos a vizinhos e valores imputados”.

### 8. Conclusão

**Complemento para revisar.** Na avaliação com `K = 9`, a random forest manteve 99,75% de acurácia após restringir a imputação ao treino de cada dobra. Ao variar apenas a divisão dos dados, a acurácia para esse `K` ficou entre 99,50% e 99,75%. Eu faria a padronização e a imputação dentro de cada dobra, controlaria as sementes e reportaria a variação das métricas juntamente com as contagens de erros. Também verificaria se o perfil dos vizinhos completos aproxima os preenchimentos de uma das classes.

**No resultado.ipynb.** Seções “Possível vazamento de dados na validação cruzada” e “7.2” a “7.4”.


### Uso de inteligência artificial

Utilizei o ChatGPT para gerar código e auxiliar na revisão e organização dos textos em Markdown, facilitando a leitura. Também houve sugestões de complementação dos textos, que submeti à minha revisão.

Todos os textos e células passaram pela minha aprovação antes do envio. Assumo a responsabilidade pela análise e pelas respostas apresentadas neste exercício.

Fico à disposição para esclarecer qualquer dúvida.

**Gabriel Buche**
